# Phase 3 RQ1 v2 — B0 training

**This is the only stage that requires a Kaggle GPU/CUDA runtime.** Turn Internet on, attach the private dataset containing `phase3_rq1_v2_inputs.zip`, accept the gated `meta-llama/Llama-3.2-1B` terms, and add a Kaggle secret named `HF_TOKEN`.

The frozen languages are English (`en`), German (`de`), Spanish (`es`), French (`fr`), Swahili (`sw`), Arabic (`ar`), Urdu (`ur`), Bulgarian (`bg`), Russian (`ru`), and Hindi (`hi`). Run this notebook once for each seed/order pair: seeds `42`, `43`, and `44` crossed with orders `order-1` through `order-5` (15 runs total). Set `SEED_TO_RUN` and `ORDER_TO_RUN` for each run. Each run trains only one order, validates the selected XNLI data hash against the input-preparation bundle, and produces its own ZIP when that order finishes. Keep all 15 `phase3_rq1_v2_b0_seed_<seed>_<order-id>.zip` files; the analysis notebook accepts them directly. Each run needs a Kaggle GPU/CUDA runtime.

Three seeds are the requested initial v2 pilot; they do not guarantee that any prior will be statistically supported.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import uuid
from zipfile import ZIP_DEFLATED, ZipFile

REPO_URL = "https://github.com/Mani-S-2709/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    raise FileExistsError(f"Repository path exists but is not a Git clone: {REPO_ROOT}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(REPO_ROOT / "phase2" / "requirements-phase2.txt"),
], check=True)

from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

sys.path.insert(0, str(REPO_ROOT))
input_bundles = list(Path("/kaggle/input").rglob("phase3_rq1_v2_inputs.zip"))
if not input_bundles:
    input_summaries = list(Path("/kaggle/input").rglob("rq1_v2_input_summary.json"))
    if len(input_summaries) == 1:
        input_root = input_summaries[0].parent
        input_bundle = Path("/kaggle/working/phase3_rq1_v2_inputs.zip")
        with ZipFile(input_bundle, "w", compression=ZIP_DEFLATED) as archive:
            for path in sorted(input_root.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(input_root).as_posix())
        input_bundles = [input_bundle]
        print(f"Repacked the extracted input dataset as: {input_bundle}")
if len(input_bundles) != 1:
    raise FileNotFoundError(
        "Attach exactly one Kaggle dataset containing either "
"phase3_rq1_v2_inputs.zip or its extracted contents. Found ZIPs: "
+ str(input_bundles)
    )

SEED_TO_RUN = 42
if SEED_TO_RUN not in (42, 43, 44):
    raise ValueError("SEED_TO_RUN must be 42, 43, or 44.")
ORDER_TO_RUN = "order-1"  # Run order-1 through order-5 for each seed.
if ORDER_TO_RUN not in {f"order-{index}" for index in range(1, 6)}:
    raise ValueError("ORDER_TO_RUN must be order-1 through order-5.")

OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_v2_seed_outputs")
WORK_DIR = Path("/kaggle/temp/phase3_rq1_v2") / f"seed_{SEED_TO_RUN}_{uuid.uuid4().hex}"
subprocess.run([
    sys.executable, "-m", "phase3.rq1_v2_workflow", "train-order",
    "--inputs-zip", str(input_bundles[0]),
    "--seed", str(SEED_TO_RUN),
    "--order-id", ORDER_TO_RUN,
    "--work-dir", str(WORK_DIR),
    "--output-dir", str(OUTPUT_DIR),
], cwd=REPO_ROOT, check=True)

RESULT_ZIP = OUTPUT_DIR / f"phase3_rq1_v2_b0_seed_{SEED_TO_RUN}_{ORDER_TO_RUN}.zip"
if not RESULT_ZIP.is_file():
    raise FileNotFoundError(f"Validated result bundle was not created: {RESULT_ZIP}")
print(f"Download this seed's result bundle from Kaggle output files: {RESULT_ZIP}")

Cloning into '/kaggle/working/asymmetric-interference-continual-learning'...


Repacked the extracted input dataset as: /kaggle/working/phase3_rq1_v2_inputs.zip
{
  "cuda_available": true,
  "cuda_device": "Tesla T4",
  "cuda_device_count": 2,
  "cuda_total_memory_gib": 14.56,
  "gated_model_metadata_checked": true,
  "huggingface_token_present": true,
  "model_id": "meta-llama/Llama-3.2-1B",
  "package_versions": {
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
    "datasets": "5.0.1",
    "peft": "0.21.2",
    "torch": "2.11.0+cu130",
    "transformers": "5.18.0"
  },
  "python_version": "3.13.16",
  "tokenizer_pad_token_present": false
}


Generating validation split: 100%|██████████| 2490/2490 [00:00<00:00, 563586.26 examples/s]


Starting xnli-rq1-v2-phase4-roster-10lang-500x200-v2-seed-42-B0-order-1-seed-42


Tokenizing causal-LM prompt/target data: 100%|██████████| 500/500 [00:00<00:00, 2977.04 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.
100%|██████████| 32/32 [02:45<00:00,  5.16s/it]


{'loss': '1.543', 'grad_norm': '10.29', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '1.066', 'grad_norm': '10.66', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.9567', 'grad_norm': '2.915', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '165.1', 'train_samples_per_second': '3.028', 'train_steps_per_second': '0.194', 'train_loss': '1.175', 'epoch': '1'}


... (14 outputs left out)

100%|██████████| 32/32 [02:52<00:00,  5.40s/it]


{'loss': '0.9043', 'grad_norm': '7.175', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '0.7906', 'grad_norm': '5.524', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.5896', 'grad_norm': '5.417', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '172.9', 'train_samples_per_second': '2.891', 'train_steps_per_second': '0.185', 'train_loss': '0.751', 'epoch': '1'}


100%|██████████| 32/32 [03:27<00:00,  6.47s/it]


{'loss': '0.8726', 'grad_norm': '5.428', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '0.7139', 'grad_norm': '7.027', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.701', 'grad_norm': '8.198', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '207', 'train_samples_per_second': '2.415', 'train_steps_per_second': '0.155', 'train_loss': '0.7709', 'epoch': '1'}
Completed xnli-rq1-v2-phase4-roster-10lang-500x200-v2-seed-42-B0-order-1-seed-42: ACC=0.6450
{
  "completed_runs": 1,
  "data_manifest_sha256": "373852a7f1e09ab99da8a25f21023fedc1977ab1969e09d938874f7c645a721a",
  "expected_orders": [
    "order-1",
    "order-2",
    "order-3",
    "order-4",
    "order-5"
  ],
  "expected_runs": 1,
  "metrics_mean_across_orders": {
    "ACC": 0.645,
    "BWT": 0.09444444444444443,
    "F": 0.019444444444444438,
    "FWT": 0.2188888888888889
  },
  "metrics_population_std_across_orders": {
    "ACC": 0.0,
    "BWT": 0.0,
    "F": 0.0,
    "FWT": 0.0
  },
  "ord